# 03 - GARCH-Family Models

Estimate GARCH(1,1), EGARCH(1,1), and GJR-GARCH(1,1) conditional volatility.


In [1]:
import numpy as np
import pandas as pd
from arch import arch_model

df = pd.read_csv("data_nepse_returns.csv", parse_dates=["date_ad"], index_col="date_ad")
returns = df["log_return_pct"].dropna()
print(f"Model sample: {returns.index.min().date()} to {returns.index.max().date()}, n={len(returns):,}")


Model sample: 2015-01-04 to 2026-06-03, n=2,595


In [2]:
models = {
    "garch_11": arch_model(returns, mean="Constant", vol="GARCH", p=1, q=1, dist="normal", rescale=False),
    "egarch_11": arch_model(returns, mean="Constant", vol="EGARCH", p=1, q=1, dist="normal", rescale=False),
    "gjr_garch_11": arch_model(returns, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="normal", rescale=False),
}

results = {}
for name, model in models.items():
    print(f"Fitting {name}...")
    results[name] = model.fit(disp="off")

[(name, result.aic, result.bic) for name, result in results.items()]


Fitting garch_11...
Fitting egarch_11...
Fitting gjr_garch_11...


[('garch_11', 8279.39652499164, np.float64(8302.84189217404)),
 ('egarch_11', 8280.072359115824, np.float64(8303.517726298223)),
 ('gjr_garch_11', 8266.79986977294, np.float64(8296.10657875094))]

In [3]:
vol = pd.DataFrame(index=returns.index)
for name, result in results.items():
    vol[f"{name}_daily_pct"] = result.conditional_volatility
    vol[f"{name}_annualized_pct"] = result.conditional_volatility * np.sqrt(252)

vol.to_csv("data_garch_volatility_estimates.csv")
vol.tail()


,garch_11_daily_pct,garch_11_annualized_pct,egarch_11_daily_pct,egarch_11_annualized_pct,gjr_garch_11_daily_pct,gjr_garch_11_annualized_pct
date_ad,,,,,,
2026-05-26,0.935525,14.851002,0.985342,15.641821,0.907253,14.402197
2026-05-27,0.895727,14.219223,0.935379,14.848688,0.876097,13.907607
2026-06-01,0.850302,13.498118,0.859651,13.646532,0.835350,13.260763
2026-06-02,0.922999,14.652159,0.953204,15.131638,0.931326,14.784349
2026-06-03,0.929377,14.753404,0.984555,15.629331,0.924667,14.678631


In [4]:
model_table = pd.DataFrame(
    {
        name: {
            "AIC": result.aic,
            "BIC": result.bic,
            "LogLik": result.loglikelihood,
        }
        for name, result in results.items()
    }
).T

model_table.to_csv("table_garch_model_selection.csv")
model_table


,AIC,BIC,LogLik
garch_11,8279.396525,8302.841892,-4135.698262
egarch_11,8280.072359,8303.517726,-4136.036180
gjr_garch_11,8266.799870,8296.106579,-4128.399935
